# Milestone-3

In [ ]:
!pip install faiss-cpu

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 18.2 MB/s eta 0:00:00


## SETUP

In [ ]:
train = pd.read_csv('train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = bi_encoder.encode(kb, show_progress_bar=True)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("Knowledge base successfully created")

# Helper: retrieve top-k doc indices for any prompt string
def retrieve(prompt_text, k):
    emb = bi_encoder.encode([prompt_text])
    D, I = index.search(np.array(emb), k)
    return I[0].tolist(), D[0].tolist()


Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge base successfully created


## Q1 - Zero-shot classifier baseline (no RAG) on row 150


In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']),
              str(row_150['D']), str(row_150['E'])]
ans_150_text = str(row_150[row_150['answer']])

result_q1 = zs(prompt_150, candidate_labels=labels_150)
q1_score = result_q1['scores'][result_q1['labels'].index(ans_150_text)]
print(f"Q1 - baseline probability of correct option: {round(q1_score, 3)}")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Q1 - baseline probability of correct option: 0.384


## Q2 - FAISS retrieval rank of the true document for row 150 (top k=10)

In [ ]:
retrieved_indices_150, _ = retrieve(prompt_150, k=10)
if 150 in retrieved_indices_150:
    q2_rank = retrieved_indices_150.index(150) + 1
else:
    q2_rank = None  # not found in top 10
print(f"Q2 - FAISS rank of true doc (row150) in top10: {q2_rank}")


Q2 - FAISS rank of true doc (row150) in top10: 10


## Q3 - Cross-encoder rerank of those same 10 docs

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices_150]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort (index_in_kb, score) pairs by ce score descending
ranked = sorted(zip(retrieved_indices_150, ce_scores), key=lambda x: x[1], reverse=True)
ranked_indices = [r[0] for r in ranked]
q3_rank = ranked_indices.index(150) + 1 if 150 in ranked_indices else None
print(f"Q3 - Cross-encoder rank of true doc (row150): {q3_rank}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Q3 - Cross-encoder rank of true doc (row150): 1


## Q4 - Token count for row 42, top k=5 context

In [ ]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
retrieved_42_idx, _ = retrieve(prompt_42, k=5)
docs_42 = [kb[i] for i in retrieved_42_idx]
concatenated_42 = " ".join(docs_42)
rag_string_42 = f"Context: {concatenated_42} Question: {prompt_42}"

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens_42 = bert_tokenizer(rag_string_42, truncation=False)['input_ids']
q4_token_count = len(tokens_42)
print(f"Q4 - total token count: {q4_token_count}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Q4 - total token count: 216


## Q5 - RAG with the TRUE document for row 150


In [ ]:
true_doc_150 = kb[150]
rag_string_150_true = f"Context: {true_doc_150} Question: {prompt_150}"
result_q5 = zs(rag_string_150_true, candidate_labels=labels_150)
q5_score = result_q5['scores'][result_q5['labels'].index(ans_150_text)]
print(f"Q5 - RAG (true doc) probability of correct option: {round(q5_score, 3)}")


Q5 - RAG (true doc) probability of correct option: 0.989


## Q6 - Adversarial RAG: force context to KB index 999 (unrelated fact)

In [ ]:
adversarial_doc = kb[999]
rag_string_150_adv = f"Context: {adversarial_doc} Question: {prompt_150}"
result_q6 = zs(rag_string_150_adv, candidate_labels=labels_150)
q6_score = result_q6['scores'][result_q6['labels'].index(ans_150_text)]
print(f"Q6 - Adversarial RAG probability of correct option: {round(q6_score, 3)}")


Q6 - Adversarial RAG probability of correct option: 0.529


## Q7 - Hit Rate @5 for first 100 rows

In [ ]:
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])
    retrieved_idx, _ = retrieve(prompt_i, k=5)
    retrieved_docs = [kb[j] for j in retrieved_idx]
    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

q7_hit_rate = round(hits / 100 * 100, 1)
print(f"Q7 - Hit Rate over 100 rows: {q7_hit_rate}%")


Q7 - Hit Rate over 100 rows: 73.0%


## Q8 - Full RAG pipeline

In [ ]:
def average_precision_at_3(ranked_letters, correct_letter):
    """ranked_letters: list of top-3 letters ordered highest->lowest prob."""
    for rank, letter in enumerate(ranked_letters, start=1):
        if letter == correct_letter:
            return 1.0 / rank
    return 0.0

ap_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_letter = row['answer']

    letters = ['A', 'B', 'C', 'D', 'E']
    option_texts = [str(row[l]) for l in letters]
    text_to_letter = dict(zip(option_texts, letters))

    # Retrieve top 5
    retrieved_idx, _ = retrieve(prompt_i, k=5)
    docs_5 = [kb[j] for j in retrieved_idx]

    # Rerank with cross-encoder, take single best doc
    pairs_i = [[prompt_i, d] for d in docs_5]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_5[int(np.argmax(ce_scores_i))]

    # Augment
    rag_string_i = f"Context: {best_doc} Question: {prompt_i}"

    # Predict
    result_i = zs(rag_string_i, candidate_labels=option_texts)
    # result_i['labels'] already sorted by score descending
    ranked_letters_i = [text_to_letter[lbl] for lbl in result_i['labels'][:3]]

    ap = average_precision_at_3(ranked_letters_i, correct_letter)
    ap_scores.append(ap)

q8_map3 = round(float(np.mean(ap_scores)), 3)
print(f"Q8 - MAP@3 over 20 rows: {q8_map3}")

Q8 - MAP@3 over 20 rows: 0.975
